# BirdCLEF+ 2025 - Pretrained models: GPU (torchaudio) mel vs A1's librosa mel

**T4 GPU, internet off, no training.** This notebook checks whether a torchaudio log-mel on CUDA reproduces A1's exact librosa input, so that a student trainer can compute its inputs on the GPU.

**Samples:** 50 train clips (seed 42, the baseline's `train_index.csv`, first 5 s window of each) and 50 soundscape windows (seed 42, random file of `train_soundscapes` and random window index). Decode = A1's `decode` (mono, resample to 32 kHz if needed, end padded to whole 5 s windows).

**Reference:** A1's librosa path: `melspectrogram` -> `power_to_db(ref=np.max, top_db)` -> `to_input` (cfg from `a1-v2s-train`).

**Candidate:** `TorchA1Mel` (torchaudio `MelSpectrogram` on CUDA, same pad mode as the installed librosa, Slaney mel scale and norm, periodic Hann) -> the same dB rule and `to_input` in torch.

**Reported:** max / mean abs difference of the dB mel and of the model input, plus A1 sigmoid probabilities on both inputs (eval, float32) and the fraction of windows whose argmax changes. The verdict is `NEAR_IDENTICAL` if the input and probability max abs diff are both <= 1e-3. Output: `mel_check.json`.

In [ ]:
import os, time
import torch

T_NOTEBOOK = time.time()
SMOKE = os.environ.get("M3_SMOKE_TEST") == "1"     # local CPU test only, never set on Kaggle
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("torch.cuda.get_device_name(0):", torch.cuda.get_device_name(0))
elif SMOKE:
    print("M3_SMOKE_TEST=1: no GPU, running the local CPU test")
else:
    raise RuntimeError("No GPU found. Enable the T4 accelerator; this notebook never falls back to CPU.")

In [ ]:
import glob, inspect, json
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torchaudio
import timm
from scipy.signal import resample_poly

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu (smoke test)"
torch.backends.cuda.matmul.allow_tf32 = False      # full float32 everywhere (T4 has no TF32 anyway)
torch.backends.cudnn.allow_tf32 = False
OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))
SEED = 42
N_TRAIN = int(os.environ.get("M3_N_TRAIN", "50"))
N_SOUND = int(os.environ.get("M3_N_SOUND", "50"))
TOL = 1e-3
BUDGET_S = float(os.environ.get("M3_BUDGET_S", 30 * 60))
SOURCE_KERNEL = "sc4000-birdclef-m3-a1-v2s-train"

def check_budget(projected_s, where):
    msg = f"{where}: projected notebook time {projected_s:.0f}s = {projected_s / 60:.1f} min (limit {BUDGET_S / 60:.0f} min)"
    print("TIMING GUARD |", msg, flush=True)
    if projected_s > BUDGET_S:
        raise RuntimeError(f"Projected time is over the limit. {msg}")

def gpu_sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

def find_dir(marker, env):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    if len(hits) != 1:
        raise FileNotFoundError(f"expected exactly one {marker} under /kaggle/input, found {hits}")
    return Path(hits[0]).parent

def find_model_dir(kernel):
    if os.environ.get("M3_MODEL_DIR"):      # local tests only
        return Path(os.environ["M3_MODEL_DIR"])
    hits = [h for h in glob.glob("/kaggle/input/**/model.pth", recursive=True) if kernel in h]
    if len(hits) != 1:
        raise RuntimeError(f"expected one model.pth under kernel {kernel}, found {hits}")
    return Path(hits[0]).parent

DATA = find_dir("birdclef-2025/sample_submission.csv", "M3_DATA_DIR")
MELS = find_dir("train_index.csv", "M3_MELS_DIR")
MODEL_DIR = find_model_dir(SOURCE_KERNEL)
CFG = json.load(open(MODEL_DIR / "cfg.json"))
species = json.load(open(MODEL_DIR / "species.json"))
assert species == list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:]), "species order differs from sample_submission"
assert CFG["sr"] == 32000 and CFG["duration"] == 5, (CFG["sr"], CFG["duration"])
if not CFG["model"].startswith("tf_efficientnetv2_s"):
    raise RuntimeError(f"expected tf_efficientnetv2_s, cfg.json has {CFG['model']}")

# The pad mode librosa uses for center=True in the installed version (it changed from "reflect" to "constant" in 0.10).
LIBROSA_PAD_MODE = inspect.signature(librosa.feature.melspectrogram).parameters["pad_mode"].default
LIBROSA_STFT_PAD_MODE = inspect.signature(librosa.stft).parameters["pad_mode"].default
assert LIBROSA_PAD_MODE == LIBROSA_STFT_PAD_MODE, (LIBROSA_PAD_MODE, LIBROSA_STFT_PAD_MODE)
VERSIONS = dict(librosa=librosa.__version__, torch=torch.__version__, torchaudio=torchaudio.__version__, timm=timm.__version__)
print("Data:", DATA, "| M1 index:", MELS, "| model:", MODEL_DIR, "| device:", DEVICE_NAME)
print("versions:", VERSIONS, "| librosa pad_mode:", LIBROSA_PAD_MODE)
print("cfg:", {k: CFG[k] for k in ("sr", "n_fft", "hop_length", "n_mels", "fmin", "fmax", "norm", "norm_top_db")})

## Reference: A1's librosa path (copied from `ns_a1_all`)

In [ ]:
WIN, TOP_DB = CFG["sr"] * CFG["duration"], float(CFG["norm_top_db"])

def to_input(db):
    # Same as to_input in the A1 notebooks. For norm "v2" this equals M1's (db + 80) / 80.
    db = np.maximum(db - db.max(), -TOP_DB)
    if CFG["norm"] == "zscore_minmax":
        x = (db - db.mean()) / (db.std() + 1e-6)
        return ((x - x.min()) / (x.max() - x.min() + 1e-6)).astype(np.float32)
    return ((db + TOP_DB) / TOP_DB).astype(np.float32)

def decode(path):
    # The fast-inference decode plus resampling (as ns_a1_all).
    y, sr = sf.read(str(path), dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != CFG["sr"]:
        y = resample_poly(y, CFG["sr"], sr).astype(np.float32)
    if len(y) == 0 or not np.isfinite(y).all():
        raise ValueError(f"empty or non-finite audio: {Path(path).name}")
    n_win = max(1, int(np.ceil(len(y) / WIN)))
    y = np.pad(y, (0, n_win * WIN - len(y)))
    return y.reshape(n_win, WIN)

def librosa_db(batch):
    power = librosa.feature.melspectrogram(
        y=batch, sr=CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"])
    return np.stack([librosa.power_to_db(m, ref=np.max, top_db=TOP_DB) for m in power])

def windows_to_inputs(batch):
    # = A1's windows_to_inputs (one melspectrogram call, per-window ref and normalisation).
    return np.stack([to_input(d) for d in librosa_db(batch)])

## Candidate: torch / torchaudio mel (self-contained; copy this cell into the student trainer)
Call it outside any `autocast` block: the mel must stay in float32.

In [ ]:
import torch
import torchaudio


class TorchA1Mel(torch.nn.Module):
    """(B, sr * duration) float32 waveforms -> (B, n_mels, frames) A1 model inputs, on any device.

    Reproduces librosa.feature.melspectrogram(center=True, power=2, Slaney mel, periodic Hann)
    -> librosa.power_to_db(ref=np.max, amin=1e-10, top_db) -> A1's to_input, all per window.
    pad_mode must be the installed librosa's default ("constant" for librosa >= 0.10, "reflect" before).
    """

    def __init__(self, cfg, pad_mode):
        super().__init__()
        self.top_db = float(cfg["norm_top_db"])
        self.norm = cfg["norm"]
        self.amin = 1e-10
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=cfg["sr"], n_fft=cfg["n_fft"], win_length=cfg["n_fft"], hop_length=cfg["hop_length"],
            f_min=float(cfg["fmin"]), f_max=float(cfg["fmax"]), n_mels=cfg["n_mels"],
            window_fn=torch.hann_window,         # periodic=True by default, like scipy get_window("hann", fftbins=True)
            power=2.0, normalized=False, center=True, pad_mode=pad_mode, onesided=True,
            norm="slaney", mel_scale="slaney")

    def db(self, wav):
        """power_to_db(S, ref=np.max, amin=1e-10, top_db) per window."""
        s = self.mel(wav.float())
        dims = (1, 2)
        log_spec = 10.0 * torch.log10(torch.clamp(s, min=self.amin))
        log_spec = log_spec - 10.0 * torch.log10(torch.clamp(s.amax(dim=dims, keepdim=True), min=self.amin))
        return torch.maximum(log_spec, log_spec.amax(dim=dims, keepdim=True) - self.top_db)

    def to_input(self, db):
        """A1's to_input per window (np.std is the population std, hence correction=0)."""
        dims = (1, 2)
        db = torch.clamp(db - db.amax(dim=dims, keepdim=True), min=-self.top_db)
        if self.norm == "zscore_minmax":
            x = (db - db.mean(dim=dims, keepdim=True)) / (db.std(dim=dims, keepdim=True, correction=0) + 1e-6)
            lo, hi = x.amin(dim=dims, keepdim=True), x.amax(dim=dims, keepdim=True)
            return (x - lo) / (hi - lo + 1e-6)
        return (db + self.top_db) / self.top_db

    def forward(self, wav):
        return self.to_input(self.db(wav))

## Samples

In [ ]:
t0 = time.time()
index = pd.read_csv(MELS / "train_index.csv")
if "filename" in index.columns:
    filenames = index["filename"].astype(str).tolist()
else:
    # Only safe when M1's index is row-for-row train.csv (same rule as a2-perch-embed).
    train_csv = pd.read_csv(DATA / "train.csv")
    same = len(train_csv) == len(index) and (train_csv["primary_label"].astype(str).values
                                             == index["primary_label"].astype(str).values).all()
    if not same:
        raise RuntimeError(f"train_index.csv has no filename column and does not align with train.csv; columns: {list(index.columns)}")
    filenames = train_csv["filename"].astype(str).tolist()
rng = np.random.default_rng(SEED)
train_pick = np.sort(rng.choice(len(filenames), size=min(N_TRAIN, len(filenames)), replace=False))
train_names = [filenames[i] for i in train_pick]
train_wav = np.stack([decode(DATA / "train_audio" / n)[0] for n in train_names])     # first 5 s window
check_budget(time.time() - T_NOTEBOOK + 60 + 3 * (time.time() - t0), "after decoding the train clips")

sound_names_all = sorted(p.name for p in (DATA / "train_soundscapes").glob("*.ogg"))
assert sound_names_all, "no train_soundscapes"
rng = np.random.default_rng(SEED)
sound_files = rng.choice(len(sound_names_all), size=N_SOUND, replace=len(sound_names_all) < N_SOUND)
sound_wav, sound_keys = [], []
for j in sound_files:
    w = decode(DATA / "train_soundscapes" / sound_names_all[j])
    k = int(rng.integers(len(w)))
    sound_wav.append(w[k]); sound_keys.append((sound_names_all[j], k))
sound_wav = np.stack(sound_wav)
DECODE_S = time.time() - t0
print(f"train {train_wav.shape}, soundscape {sound_wav.shape}, decode {DECODE_S:.1f}s", flush=True)
print("first train clips:", train_names[:3], "| first soundscape windows:", sound_keys[:3])
check_budget(time.time() - T_NOTEBOOK + 120, "after decoding")

## Compare inputs and A1 predictions

In [ ]:
mel_gpu = TorchA1Mel(CFG, LIBROSA_PAD_MODE).to(DEVICE)
model = timm.create_model(CFG["model"], pretrained=False, in_chans=1, num_classes=len(species))
model.load_state_dict(torch.load(MODEL_DIR / "model.pth", map_location="cpu"))
model = model.to(DEVICE).eval()
assert not any(m.training for m in model.modules())

def probs(x):
    out = []
    with torch.inference_mode():
        for i in range(0, len(x), 25):
            out.append(torch.sigmoid(model(x[i:i + 25][:, None].float())).float().cpu().numpy())
    return np.concatenate(out)

def diff(a, b):
    d = np.abs(a.astype(np.float64) - b.astype(np.float64))
    return dict(max_abs=float(d.max()), mean_abs=float(d.mean()))

results, timing = {}, {}
for kind, wav in (("train_clips", train_wav), ("soundscape_windows", sound_wav)):
    t = time.time()
    ref_db = librosa_db(wav)
    ref_in = windows_to_inputs(wav)
    t_ref = time.time() - t
    with torch.inference_mode():
        w = torch.from_numpy(wav).to(DEVICE)
        _ = mel_gpu(w[:2]); gpu_sync()                 # warm-up (cuFFT plan)
        t = time.time()
        cand_db_t = mel_gpu.db(w)
        cand_in_t = mel_gpu.to_input(cand_db_t)
        gpu_sync()
        t_cand = time.time() - t
    cand_db, cand_in = cand_db_t.cpu().numpy(), cand_in_t.cpu().numpy()
    assert ref_db.shape == cand_db.shape and ref_in.shape == cand_in.shape, (ref_db.shape, cand_db.shape, ref_in.shape, cand_in.shape)
    assert ref_in.dtype == np.float32 and cand_in.dtype == np.float32
    p_ref = probs(torch.from_numpy(ref_in).to(DEVICE))
    p_cand = probs(cand_in_t)
    r = dict(n=int(len(wav)), shape=list(ref_in.shape), db_before_to_input=diff(ref_db, cand_db), model_input=diff(ref_in, cand_in),
             probability=diff(p_ref, p_cand), argmax_changed_fraction=float((p_ref.argmax(1) != p_cand.argmax(1)).mean()),
             mean_prob_ref=float(p_ref.mean()), mean_prob_cand=float(p_cand.mean()))
    results[kind] = r
    timing[kind] = dict(librosa_cpu_s=t_ref, torch_device_s=t_cand)
    print(f"[{kind}] n={r['n']} shape {tuple(r['shape'])} | dB max {r['db_before_to_input']['max_abs']:.3e} mean {r['db_before_to_input']['mean_abs']:.3e} | "
          f"input max {r['model_input']['max_abs']:.3e} mean {r['model_input']['mean_abs']:.3e} | "
          f"prob max {r['probability']['max_abs']:.3e} mean {r['probability']['mean_abs']:.3e} | argmax changed {r['argmax_changed_fraction']:.3f} | "
          f"librosa {t_ref:.2f}s vs torch {t_cand:.3f}s", flush=True)
    check_budget(time.time() - T_NOTEBOOK + 60, f"after {kind}")

input_max = max(r["model_input"]["max_abs"] for r in results.values())
prob_max = max(r["probability"]["max_abs"] for r in results.values())
verdict = "NEAR_IDENTICAL" if input_max <= TOL and prob_max <= TOL else "NOT_IDENTICAL"
print(f"VERDICT | {verdict} | model-input max abs diff {input_max:.3e} | probability max abs diff {prob_max:.3e} | tolerance {TOL}")

In [ ]:
total = time.time() - T_NOTEBOOK
report = dict(verdict=verdict, tolerance=TOL, model_input_max_abs=input_max, probability_max_abs=prob_max, results=results,
              librosa_pad_mode=LIBROSA_PAD_MODE, versions=VERSIONS, device=DEVICE_NAME, model_dir=str(MODEL_DIR), seed=SEED,
              cfg={k: CFG[k] for k in ("sr", "duration", "n_fft", "hop_length", "n_mels", "fmin", "fmax", "norm", "norm_top_db", "model")},
              torch_mel=dict(transform="torchaudio.transforms.MelSpectrogram", window="hann periodic", center=True, pad_mode=LIBROSA_PAD_MODE,
                             power=2.0, mel_scale="slaney", norm="slaney", db="10*log10(max(S,1e-10)) - 10*log10(max(max S,1e-10)), floor at max - top_db"),
              samples=dict(train_clips=train_names, soundscape_windows=[dict(filename=n, win=k) for n, k in sound_keys]),
              seconds=dict(decode=DECODE_S, mel=timing, notebook=total))
with open(OUT / "mel_check.json", "w") as f:
    json.dump(report, f, indent=1)
print(f"TIMING | decode {DECODE_S:.1f}s | " + " | ".join(f"{k}: librosa {v['librosa_cpu_s']:.2f}s, torch {v['torch_device_s']:.3f}s" for k, v in timing.items())
      + f" | notebook {total:.0f}s = {total / 60:.1f} min | device {DEVICE_NAME}")